<a href="https://colab.research.google.com/github/RENJIRO23/Big-Data-Processing/blob/main/phase3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Pyspark

In [ ]:
!pip install pyspark

In [ ]:
!pip install imbalanced-learn

# Import Pyspark

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import StringType, StructType, StructField, LongType, DoubleType
import builtins

# Inisialisasi Spark Session

In [ ]:
spark = SparkSession.builder \
    .appName("Brooklyn Air Quality Analytics") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

#Ingenstion Layer | Edited: 17/06/2026

In [ ]:
print("=== DATA INGESTION LAYER ===")

DATA_SOURCE = "OpenAQ"

print(f"Source Dataset : {DATA_SOURCE}")
print("Ingestion Type : Batch Ingestion")
print("Processing Engine : Apache Spark")
print("Storage Layer : HDFS (Production) / Local CSV (Development)")

=== DATA INGESTION LAYER ===
Source Dataset : OpenAQ
Ingestion Type : Batch Ingestion
Processing Engine : Apache Spark
Storage Layer : HDFS (Production) / Local CSV (Development)


# Import Dataset (Sekarang ada opsi HDFS) | Edited: 17/06/2026

In [ ]:
USE_HDFS = False

if USE_HDFS:
    measurements_path = "hdfs:///airquality/raw/measurements.csv"
    stations_path = "hdfs:///airquality/raw/stations.csv"
else:
    measurements_path = "measurements.csv"
    stations_path = "stations.csv"

df_measurements = spark.read.csv(
    measurements_path,
    header=True,
    inferSchema=True
)

df_stations = spark.read.csv(
    stations_path,
    header=True,
    inferSchema=True
)

print("Measurements Source:", measurements_path)
print("Stations Source:", stations_path)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/measurements.csv. SQLSTATE: 42K03

#Arsitektur | Edited: 17/06/2026

OpenAQ Dataset
      ↓
Data Ingestion
      ↓
HDFS / Cloudera
      ↓
Spark ETL
      ↓
Feature Engineering
      ↓
ADASYN
      ↓
Random Forest
      ↓
Cross Validation
      ↓
Evaluation

#Justifikasi Batch Processing | Edited: 17/06/2026

Dataset yang digunakan merupakan historical air quality dataset.

Karena analisis dilakukan terhadap data historis dan tidak
membutuhkan prediksi real-time, batch processing dipilih
sebagai pendekatan yang lebih sesuai dibanding streaming.

#Kalau Data Bertambah | Edited: 17/06/2026

In [ ]:
print("=== INCREMENTAL INGESTION SIMULATION ===")

current_records = df_measurements.count()

incoming_records = int(current_records * 0.1)

print("Current Records :", current_records)
print("Estimated Incoming Records :", incoming_records)

print(
    "Pipeline supports incremental ingestion when new files arrive."
)

In [ ]:
def null_value_report(df, dataset_name):
    total_rows = df.count()

    null_exprs = []

    for column_name, dtype in df.dtypes:
        if dtype in ["double", "float"]:
            null_condition = F.col(column_name).isNull() | F.isnan(F.col(column_name))
        else:
            null_condition = F.col(column_name).isNull()

        null_exprs.append(
            F.sum(F.when(null_condition, 1).otherwise(0)).alias(column_name)
        )

    null_counts = df.select(null_exprs).collect()[0].asDict()

    report_data = []
    for column_name, null_count in null_counts.items():
        null_percentage = (null_count / total_rows * 100) if total_rows > 0 else 0

        report_data.append((
            dataset_name,
            column_name,
            int(null_count),
            float(builtins.round(null_percentage, 2))
        ))

    schema = StructType([
        StructField("dataset", StringType(), True),
        StructField("column", StringType(), True),
        StructField("null_count", LongType(), True),
        StructField("null_percentage", DoubleType(), True)
    ])

    return spark.createDataFrame(report_data, schema)

In [ ]:
print("NULL VALUE REPORT: MEASUREMENTS DATASET")

null_report_measurements = null_value_report(df_measurements, "measurements")

null_report_measurements.orderBy(
    F.desc("null_percentage")
).show(truncate=False)

#Missing Value Justification | Edited: 17/06/2026



In [ ]:
high_missing = null_report_measurements.filter(
    F.col("null_percentage") > 50
)

print("Columns with >50% missing values")
high_missing.show(truncate=False)

In [ ]:
print("NULL VALUE REPORT: STATIONS DATASET")

null_report_stations = null_value_report(df_stations, "stations")

null_report_stations.orderBy(
    F.desc("null_percentage")
).show(truncate=False)

# Data Cleaning

In [ ]:
df_measurements.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_measurements.columns
]).show()

In [ ]:
df_clean = df_measurements.dropDuplicates()
df_clean = df_clean.dropna(subset=["datetimeLocal", "value", "location_id"])
df_clean = df_clean.filter(df_clean["value"] >= 0)
df_clean.show()
print("Jumlah baris setelah cleaning:", df_clean.count())

# Data Transformation & Feature Engineering

In [ ]:
drop_col = ["parameter", "unit", "country_iso", "timezone", "provider", "owner_name", "isMobile", "isMonitor", "datetimeUtc"]
df_clean = df_clean.withColumn("hour", hour(col("datetimeLocal"))) \
                   .withColumn("day", day(col("datetimeLocal"))) \
                   .withColumn("month", month(col("datetimeLocal"))) \
                   .withColumn("year", year(col("datetimeLocal"))) \
                   .withColumn("date", date_format(col("datetimeLocal"), "yyyy-MM-dd")) \
                   .withColumn("day_name", dayname(col("datetimeLocal"))) \
                   .drop(*drop_col)
df_clean.show()

# Aggregation

In [ ]:
# Average PM2.5 per day
daily_avg = df_clean.groupBy("date").agg(round(avg("value"), 2).alias("avg_pm25"))
df_daily_avg = daily_avg.orderBy(col("avg_pm25").desc())
df_daily_avg.show()

# Category
df_clean = df_clean.withColumn(
    "pollution_level",
    when(df_clean.value <= 12, "Good")
    .when(df_clean.value <= 35, "Moderate")
    .when(col("value") <= 55.4, "Unhealthy for Sensitive Groups")
    .otherwise("Unhealthy")
)

df_clean.show()

#Export

In [ ]:
df_clean.write.mode("overwrite").csv("processed_air_quality_csv", header=True)



#Export v2 | Edited: 17/06/2026

In [ ]:
if USE_HDFS:
    output_path = "hdfs:///airquality/processed/"
else:
    output_path = "processed_air_quality_csv"

df_clean.write \
    .mode("overwrite") \
    .csv(output_path, header=True)

print("Output Path:", output_path)

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

print("--- DESCRIPTIVE ANALYTICS ---")
df_clean.select("value").summary("count", "mean", "min", "max").show()
print("Distribusi Kelas:")
df_clean.groupBy("pollution_level").count().orderBy(col("count").desc()).show()

df_ml = df_clean.dropna(subset=["hour", "day", "month", "location_name", "day_name", "latitude", "longitude"])

indexer_loc = StringIndexer(inputCol="location_name", outputCol="location_idx", handleInvalid="keep")
indexer_day = StringIndexer(inputCol="day_name", outputCol="day_idx", handleInvalid="keep")
indexer_label = StringIndexer(inputCol="pollution_level", outputCol="label", handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=["hour", "day", "month", "location_idx", "day_idx", "latitude", "longitude"],
    outputCol="features"
)

#Baseline RF (sebelum ADASYN)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=10)

pipeline = Pipeline(stages=[indexer_loc, indexer_day, indexer_label, assembler, rf])

train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)


model = pipeline.fit(train_data)
predictions = model.transform(test_data)

evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

accuracy = evaluator_acc.evaluate(predictions)
f1_score = evaluator_f1.evaluate(predictions)

print("--- EVALUATION METRICS ---")
print(f"Accuracy : {accuracy:.4f}")
print(f"F1 Score : {f1_score:.4f}")

In [ ]:
import numpy as np
from imblearn.over_sampling import ADASYN
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import StructType, StructField, DoubleType

In [ ]:
preprocess_pipeline = Pipeline(stages=[
    indexer_loc,
    indexer_day,
    indexer_label,
    assembler
])

# Fit hanya pada train_data untuk mencegah data leakage
preprocess_model = preprocess_pipeline.fit(train_data)

train_prepared = preprocess_model.transform(train_data).select(
    "features",
    "label",
    "pollution_level"
)

test_prepared = preprocess_model.transform(test_data).select(
    "features",
    "label",
    "pollution_level"
)

label_indexer_model = preprocess_model.stages[2]

label_mapping_data = [
    (float(index), label_name)
    for index, label_name in enumerate(label_indexer_model.labels)
]

label_mapping_df = spark.createDataFrame(
    label_mapping_data,
    ["label", "pollution_level"]
)

train_pd = train_prepared.select("features", "label").toPandas()

X_train = np.vstack(
    train_pd["features"].apply(lambda x: x.toArray()).values
)

y_train = train_pd["label"].astype(int).values

print("SHAPE DATA SEBELUM ADASYN")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("DISTRIBUSI LABEL SEBELUM ADASYN")
unique_before, counts_before = np.unique(y_train, return_counts=True)

for label, count in zip(unique_before, counts_before):
    print(f"Label {label}: {count}")

In [ ]:
min_class_count = int(counts_before.min())
n_neighbors = builtins.min(5, min_class_count - 1)

print("Minimum class count:", min_class_count)
print("ADASYN n_neighbors:", n_neighbors)

adasyn = ADASYN(
    sampling_strategy="auto",
    n_neighbors=n_neighbors,
    random_state=42
)

X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)

print("SHAPE DATA SESUDAH ADASYN")
print("X_train_adasyn shape:", X_train_adasyn.shape)
print("y_train_adasyn shape:", y_train_adasyn.shape)

print("DISTRIBUSI LABEL SESUDAH ADASYN")
unique_after, counts_after = np.unique(y_train_adasyn, return_counts=True)

for label, count in zip(unique_after, counts_after):
    print(f"Label {label}: {count}")

In [ ]:
adasyn_rows = [
    (Vectors.dense(X_train_adasyn[i].tolist()), float(y_train_adasyn[i]))
    for i in range(len(y_train_adasyn))
]

adasyn_schema = StructType([
    StructField("features", VectorUDT(), False),
    StructField("label", DoubleType(), False)
])

train_adasyn_spark = spark.createDataFrame(
    adasyn_rows,
    schema=adasyn_schema
)

print("DISTRIBUSI LABEL HASIL ADASYN DALAM SPARK")
train_adasyn_spark.groupBy("label").count().orderBy(col("label")).show(truncate=False)

In [ ]:
rf_adasyn = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=10,
    seed=42
)

model_adasyn = rf_adasyn.fit(train_adasyn_spark)

predictions_adasyn = model_adasyn.transform(test_prepared)

accuracy_adasyn = evaluator_acc.evaluate(predictions_adasyn)
f1_score_adasyn = evaluator_f1.evaluate(predictions_adasyn)

print("EVALUATION METRICS AFTER ADASYN")
print(f"Accuracy : {accuracy_adasyn:.4f}")
print(f"F1 Score : {f1_score_adasyn:.4f}")

print("CONFUSION MATRIX AFTER ADASYN")
predictions_adasyn.groupBy("label", "prediction") \
    .count() \
    .orderBy("label", "prediction") \
    .show(truncate=False)

In [ ]:
rf_tuned_adasyn = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    seed=42
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

param_grid_adasyn = ParamGridBuilder() \
    .addGrid(rf_tuned_adasyn.numTrees, [50, 100, 150]) \
    .addGrid(rf_tuned_adasyn.maxDepth, [5, 10, 15]) \
    .addGrid(rf_tuned_adasyn.minInstancesPerNode, [1, 3, 5]) \
    .build()

# Cross Validation 10-fold
cv_adasyn = CrossValidator(
    estimator=rf_tuned_adasyn,
    estimatorParamMaps=param_grid_adasyn,
    evaluator=evaluator_f1,
    numFolds=10,
    seed=42,
    parallelism=2
)

cv_model_adasyn = cv_adasyn.fit(train_adasyn_spark)

predictions_cv_adasyn = cv_model_adasyn.transform(test_prepared)

accuracy_cv_adasyn = evaluator_acc.evaluate(predictions_cv_adasyn)
f1_score_cv_adasyn = evaluator_f1.evaluate(predictions_cv_adasyn)

print("EVALUATION METRICS: RANDOM FOREST + ADASYN + TUNING + 10-FOLD CV")
print(f"Accuracy : {accuracy_cv_adasyn:.4f}")
print(f"F1 Score : {f1_score_cv_adasyn:.4f}")

print("CONFUSION MATRIX: RANDOM FOREST + ADASYN + TUNING")
predictions_cv_adasyn.groupBy("label", "prediction") \
    .count() \
    .orderBy("label", "prediction") \
    .show(truncate=False)

# Perbaikan RF + ADASYN + Hyperparameter Tuning + 10-Fold CV

# Edited: 17/06/2026


In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

rf_tuned_adasyn = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    seed=42
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

# Hyperparameter grid — 3 parameter × 3 nilai = 27 kombinasi × 10 fold = 270 jobs
param_grid_adasyn = ParamGridBuilder() \
    .addGrid(rf_tuned_adasyn.numTrees, [50, 100, 150]) \
    .addGrid(rf_tuned_adasyn.maxDepth, [5, 10, 15]) \
    .addGrid(rf_tuned_adasyn.minInstancesPerNode, [1, 3, 5]) \
    .build()

print(f"Total kombinasi hyperparameter: {len(param_grid_adasyn)}")
print("Parameter yang di-tuning:")
print("  - numTrees      : [50, 100, 150]  → jumlah pohon dalam forest")
print("  - maxDepth      : [5, 10, 15]     → kedalaman max tiap pohon")
print("  - minInstancesPerNode : [1, 3, 5] → min sampel per daun (kontrol overfitting)")

# Cross Validation 10-fold
cv_adasyn = CrossValidator(
    estimator=rf_tuned_adasyn,
    estimatorParamMaps=param_grid_adasyn,
    evaluator=evaluator_f1,
    numFolds=10,           # 10-fold CV
    seed=42,
    parallelism=2
)

print("\nMemulai training: Random Forest + ADASYN + 10-Fold CV...")
cv_model_adasyn = cv_adasyn.fit(train_adasyn_spark)
print("Training selesai.")

# Evaluasi pada test set
predictions_cv_adasyn = cv_model_adasyn.transform(test_prepared)

accuracy_cv_adasyn  = evaluator_acc.evaluate(predictions_cv_adasyn)
f1_score_cv_adasyn  = evaluator_f1.evaluate(predictions_cv_adasyn)

print("\n=== EVALUATION METRICS: RF + ADASYN + Hyperparameter Tuning + 10-Fold CV ===")
print(f"Accuracy : {accuracy_cv_adasyn:.4f}")
print(f"F1 Score : {f1_score_cv_adasyn:.4f}")

print("\n=== CONFUSION MATRIX ===")
predictions_cv_adasyn.groupBy("label", "prediction") \
    .count() \
    .orderBy("label", "prediction") \
    .show(truncate=False)

In [ ]:
best_rf_adasyn = cv_model_adasyn.bestModel

best_param_map = best_rf_adasyn.extractParamMap()

print("BEST RANDOM FOREST PARAMETERS AFTER ADASYN + 10-FOLD CV")

for param, value in best_param_map.items():
    if param.name in ["numTrees", "maxDepth", "minInstancesPerNode"]:
        print(f"{param.name}: {value}")

In [ ]:
comparison_data = [
    ("Baseline Random Forest", float(accuracy), float(f1_score)),
    ("Random Forest + ADASYN", float(accuracy_adasyn), float(f1_score_adasyn)),
    ("Random Forest + ADASYN + Tuning + 10-Fold CV", float(accuracy_cv_adasyn), float(f1_score_cv_adasyn))
]

comparison_df = spark.createDataFrame(
    comparison_data,
    ["model", "accuracy", "f1_score"]
)

print("MODEL COMPARISON SUMMARY")
comparison_df.show(truncate=False)

In [ ]:
feature_columns = [
    "hour",
    "day",
    "month",
    "location_idx",
    "day_idx",
    "latitude",
    "longitude"
]

feature_importance = best_rf_adasyn.featureImportances

feature_importance_data = [
    (feature_columns[i], float(feature_importance[i]))
    for i in range(len(feature_columns))
]

feature_importance_df = spark.createDataFrame(
    feature_importance_data,
    ["feature", "importance"]
)

print("FEATURE IMPORTANCE: BEST RANDOM FOREST AFTER ADASYN + CV")
feature_importance_df.orderBy(col("importance").desc()).show(truncate=False)

Interpretation Result :
- Dataset hanya mencakup kategori Good hingga Unhealthy Sensitive Groups (Max PM2.5: 49.45).
- Skor akurasi 85% bersifat menyesatkan karena didominasi kelas mayoritas (Good mencakup 66% data). Model cenderung menebak kelas mayoritas untuk menjaga skor tetap tinggi daripada memahami pola polusi.
- Model ini hanya layak digunakan sebagai alat audit historis untuk wilayah Brooklyn pada periode tertentu, namun tidak valid untuk digunakan sebagai Sistem Peringatan Dini (Early Warning System) karena keterbatasan jangkauan data.

### VISUAL ###

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

df = pd.read_csv('/content/processed_air_quality_csv/part-00000-75417106-c0f5-4d14-ba6c-58738dffa5df-c000.csv')

df['date'] = pd.to_datetime(df['date'])
daily_trend = df.groupby('date')['value'].mean().reset_index()
full_date_range = pd.date_range(start='2025-08-01', end='2025-08-31')
daily_trend.set_index('date', inplace=True)
daily_trend = daily_trend.reindex(full_date_range).reset_index()
daily_trend.rename(columns={'index': 'date'}, inplace=True)


plt.figure(figsize=(14, 6))
sns.lineplot(data=daily_trend, x='date', y='value', marker='o', color='#2E86C1', linewidth=2.5)
plt.title('Daily Pollution Trend Analysis (August 2025)', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Average Pollution Value', fontsize=12)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
hourly_avg = df.groupby('hour')['value'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=hourly_avg, x='hour', y='value', palette='viridis')
plt.title('Hourly Average Pollution Pattern (August 2025)', fontsize=14, fontweight='bold')
plt.xlabel('Hour of the Day (0-23)', fontsize=12)
plt.ylabel('Average Pollution Value', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
pollution_counts = df['pollution_level'].value_counts()

plt.figure(figsize=(8, 8))
colors = sns.color_palette('pastel')[0:len(pollution_counts)]
plt.pie(pollution_counts,
        labels=pollution_counts.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=colors,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
plt.title('Distribution of Pollution Levels (August 2025)', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

#Best Parameter Report

# Edited: 17/06/2026

In [ ]:
best_rf_adasyn = cv_model_adasyn.bestModel

print("=== BEST HYPERPARAMETERS SETELAH TUNING + 10-FOLD CV ===")
print(f"numTrees            : {best_rf_adasyn.getNumTrees}")
print(f"maxDepth            : {best_rf_adasyn.getMaxDepth()}")
print(f"minInstancesPerNode : {best_rf_adasyn.getMinInstancesPerNode()}")

# CV scores per fold (rata-rata F1 tiap kombinasi)
print("\n=== TOP 5 KOMBINASI PARAMETER BERDASARKAN CV F1 SCORE ===")
cv_scores = cv_model_adasyn.avgMetrics  # rata-rata F1 per kombinasi
param_scores = sorted(
    zip(cv_scores, param_grid_adasyn),
    key=lambda x: -x[0]
)[:5]

for rank, (score, params) in enumerate(param_scores, 1):
    trees    = params[rf_tuned_adasyn.numTrees]
    depth    = params[rf_tuned_adasyn.maxDepth]
    min_inst = params[rf_tuned_adasyn.minInstancesPerNode]
    print(f"  #{rank} | F1={score:.4f} | numTrees={trees}, maxDepth={depth}, minInstances={min_inst}")